In [1]:
#%pip install openai
#%pip install typing-extensions
#%pip install matplotlib seaborn
#%pip install datasets transformers==4.38.2
#%pip install torch
#%pip install vllm
#%pip install --upgrade transformers

In [2]:
from experiments_vllm import *
from experiments_pretrained import *
from experiments_forced_imbalance import *
from data import *
from transformers import AutoTokenizer
import torch
from config_forced_imbalance import *
import gc

In [3]:
# Environment fixes
import os
os.environ["FLASHINFER_DISABLE_VERSION_CHECK"] = "1"

import sys, os
venv_bin = os.path.dirname(sys.executable)
if venv_bin not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = venv_bin + os.pathsep + os.environ["PATH"]

In [4]:
# Set seeds
seed = 43
torch.manual_seed(seed);

In [5]:
# Create imbalanced models
# Use full size qwen models
model_path_imbalanced = f'./vllm_benchmarking/models/{model_id_simple}-imbalance{imbalance_level_high}'
if not os.path.isdir(model_path_imbalanced):
    imbalance_pretrained_moe(model_id, imbalance_level_high, model_path_imbalanced)

In [6]:
"""
# Confirm imbalance
prompts = ["Describe the concept of Mixture in Experts in detail.",
           "Translate 'Good morning, how are you today?' into Indonesian.",
           "What is 45 multiplied by 12?",
           "Write a one-sentence definition of photosynthesis.",
           "Sort these words alphabetically: banana, apple, cherry, date.",
           "Complete the phrase: To be, or not to be, that is the..."]
#prompts *= 10

model_balanced, tokenizer = load_model(model_id, enable_bnb=enable_bnb)
probe = MoEProbeQwen(model_balanced)
for prompt in prompts:
    response, probs, active_experts = chat_generate(model_balanced, tokenizer, probe,
                                                    prompt=prompt, max_new_tokens=max_new_tokens, clear_probe=False,
                                                    prompt_formatted=False)
# Clean up memory
del model_balanced
gc.collect()
torch.cuda.empty_cache()

probe.plot_loadbalance(router_id=0)
"""

'\n# Confirm imbalance\nprompts = ["Describe the concept of Mixture in Experts in detail.",\n           "Translate \'Good morning, how are you today?\' into Indonesian.",\n           "What is 45 multiplied by 12?",\n           "Write a one-sentence definition of photosynthesis.",\n           "Sort these words alphabetically: banana, apple, cherry, date.",\n           "Complete the phrase: To be, or not to be, that is the..."]\n#prompts *= 10\n\nmodel_balanced, tokenizer = load_model(model_id, enable_bnb=enable_bnb)\nprobe = MoEProbeQwen(model_balanced)\nfor prompt in prompts:\n    response, probs, active_experts = chat_generate(model_balanced, tokenizer, probe,\n                                                    prompt=prompt, max_new_tokens=max_new_tokens, clear_probe=False,\n                                                    prompt_formatted=False)\n# Clean up memory\ndel model_balanced\ngc.collect()\ntorch.cuda.empty_cache()\n\nprobe.plot_loadbalance(router_id=0)\n'

In [7]:
"""
if enable_bnb:
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)
    model_imbalanced = AutoModelForCausalLM.from_pretrained(model_path_imbalanced,
                                                            device_map="auto",
                                                            quantization_config=quantization_config)
else:
    model_imbalanced = AutoModelForCausalLM.from_pretrained(model_path_imbalanced, device_map="auto")
    
probe = MoEProbeQwen(model_imbalanced)
tokenizer = AutoTokenizer.from_pretrained(model_path_imbalanced)
for prompt in prompts:
    response, probs, active_experts = chat_generate(model_imbalanced, tokenizer, probe,
                                                    prompt=prompt, max_new_tokens=max_new_tokens, clear_probe=False,
                                                    prompt_formatted=False)
# Clean up memory
del model_imbalanced
gc.collect()
torch.cuda.empty_cache()

probe.plot_loadbalance(router_id=0)
"""

'\nif enable_bnb:\n    quantization_config = BitsAndBytesConfig(load_in_8bit=True)\n    model_imbalanced = AutoModelForCausalLM.from_pretrained(model_path_imbalanced,\n                                                            device_map="auto",\n                                                            quantization_config=quantization_config)\nelse:\n    model_imbalanced = AutoModelForCausalLM.from_pretrained(model_path_imbalanced, device_map="auto")\n\nprobe = MoEProbeQwen(model_imbalanced)\ntokenizer = AutoTokenizer.from_pretrained(model_path_imbalanced)\nfor prompt in prompts:\n    response, probs, active_experts = chat_generate(model_imbalanced, tokenizer, probe,\n                                                    prompt=prompt, max_new_tokens=max_new_tokens, clear_probe=False,\n                                                    prompt_formatted=False)\n# Clean up memory\ndel model_imbalanced\ngc.collect()\ntorch.cuda.empty_cache()\n\nprobe.plot_loadbalance(router_id=0)\n'

In [8]:
# Use MMLU questions
dataset = get_data_mmlu(n_samples=n_samples, shuffle_seed=seed)
mmlu_prompts, subjects, questions = format_prompts_mmlu(dataset)

Streaming cais/mmlu (all) (samples: 100)...


In [9]:
# Make trace directories
if not os.path.isdir(trace_path_balanced):
    os.mkdir(trace_path_balanced)
if not os.path.isdir(trace_path_imbalanced):
    os.mkdir(trace_path_imbalanced)

In [10]:
# Run experiment over generalized MMLU questions
model_path_balanced = model_id
results_balanced = await measure_vllm_throughput(model_path_balanced,
                                                           mmlu_prompts,
                                                           seed=seed,
                                                           max_new_tokens=max_new_tokens,
                                                           max_model_len=max_model_len,
                                                           batch_size=batch_size,
                                                           concurrency_limit=batch_size*4,
                                                           gpu_memory_utilization=gpu_memory_utilization,
                                                           n_gpus=n_gpus,
                                                           n_warmup_samples=n_warmup_samples,
                                                           print_output=False,
                                                           enable_expert_parallel=enable_expert_parallel,
                                                           enable_prefix_caching=enable_prefix_caching,
                                                           enable_bnb=enable_bnb,
                                                           trace_dir=trace_path_balanced)

Starting vLLM server for Qwen/Qwen1.5-MoE-A2.7B-Chat...
Waiting for server to initialize ...
WARNING 08-25 17:44:35 [config.py:70] Support for Transformers v4 is deprecated. The Transformers v4 codepath will become unmaintained in vLLM v0.22.0 and will be removed in vLLM v0.24.0. Please upgrade to Transformers v5: pip install --upgrade transformers
WARNING 08-25 17:44:37 [profiler.py:129] Using 'torch' profiler with delay_iterations or max_iterations while ignore_frontend is False may result in high overhead.
(APIServer pid=77603) INFO 08-25 17:44:37 [utils.py:306] 
(APIServer pid=77603) INFO 08-25 17:44:37 [utils.py:306]        █     █     █▄   ▄█
(APIServer pid=77603) INFO 08-25 17:44:37 [utils.py:306]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.21.0
(APIServer pid=77603) INFO 08-25 17:44:37 [utils.py:306]   █▄█▀ █     █     █     █  model   Qwen/Qwen1.5-MoE-A2.7B-Chat
(APIServer pid=77603) INFO 08-25 17:44:37 [utils.py:306]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=77603) INFO 08-25 17:44

(Worker pid=77978) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=77978) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
(Worker pid=77930) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=77930) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
(Worker pid=77964) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will b

(Worker pid=77916) INFO 08-25 17:45:20 [pynccl.py:111] vLLM is using nccl==2.28.9
(Worker pid=77916) WARNING 08-25 17:45:21 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=77964) WARNING 08-25 17:45:21 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=77978) WARNING 08-25 17:45:21 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=77937) WARNING 08-25 17:45:21 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=77950) WARNING 08-25 17:45:21 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=77930) WARNING 08-25 17:45:21 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=77919) WARNING 08-

Loading safetensors checkpoint shards:   0% Completed | 0/8 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  12% Completed | 1/8 [00:00<00:01,  3.73it/s]
Loading safetensors checkpoint shards:  25% Completed | 2/8 [00:00<00:01,  3.44it/s]
Loading safetensors checkpoint shards:  38% Completed | 3/8 [00:00<00:01,  3.70it/s]
Loading safetensors checkpoint shards:  50% Completed | 4/8 [00:01<00:01,  3.61it/s]
Loading safetensors checkpoint shards:  62% Completed | 5/8 [00:01<00:00,  3.77it/s]
Loading safetensors checkpoint shards:  75% Completed | 6/8 [00:01<00:00,  3.65it/s]
Loading safetensors checkpoint shards:  88% Completed | 7/8 [00:01<00:00,  3.75it/s]
Loading safetensors checkpoint shards: 100% Completed | 8/8 [00:01<00:00,  4.11it/s]
(Worker_TP0_EP0 pid=77916) 


(Worker_TP0_EP0 pid=77916) INFO 08-25 17:45:30 [default_loader.py:397] Loading weights took 1.95 seconds
(Worker_TP0_EP0 pid=77916) INFO 08-25 17:45:30 [unquantized.py:345] Using MoEPrepareAndFinalizeNoDPEPModular
(Worker_TP0_EP0 pid=77916) INFO 08-25 17:45:31 [gpu_model_runner.py:4959] Model loading took 3.55 GiB memory and 7.667773 seconds
(Worker_TP0_EP0 pid=77916) WARNING 08-25 17:45:38 [fused_moe.py:1073] Using default MoE config. Performance might be sub-optimal! Config file not found at /venv/main/lib/python3.12/site-packages/vllm/model_executor/layers/fused_moe/configs/E=8,N=1408,device_name=NVIDIA_GeForce_RTX_4070.json
(Worker_TP0_EP0 pid=77916) INFO 08-25 17:45:40 [gpu_worker.py:462] Available KV cache memory: 3.24 GiB
(EngineCore pid=77764) INFO 08-25 17:45:40 [kv_cache_utils.py:1710] GPU KV cache size: 141,664 tokens
(EngineCore pid=77764) INFO 08-25 17:45:40 [kv_cache_utils.py:1711] Maximum concurrency for 2,048 tokens per request: 69.17x
(Worker_TP2_EP2 pid=77923) (Worker

...

(APIServer pid=77603) INFO:     127.0.0.1:37668 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=77603) INFO:     127.0.0.1:37682 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=77603) INFO:     127.0.0.1:37698 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=77603) INFO:     127.0.0.1:37708 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=77603) INFO:     127.0.0.1:37720 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=77603) INFO:     127.0.0.1:37728 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=77603) INFO:     127.0.0.1:37742 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=77603) INFO:     127.0.0.1:37754 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=77603) INFO:     127.0.0.1:37760 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=77603) INFO:     127.0.0.1:37768 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=77603) INFO:     127.0.0.1:37778 - "POST /v1/

(Worker_TP3_EP3 pid=77930) /venv/main/lib/python3.12/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
(Worker_TP3_EP3 pid=77930)   _warn_once(
(Worker_TP1_EP1 pid=77919) /venv/main/lib/python3.12/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
(Worker_TP1_EP1 pid=77919)   _warn_once(
(Worker_TP7_EP7 pid=77978) (Worker_TP2_EP2 pid=77923) /venv/main/lib/python3.12/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
/venv/main/lib/python3.12/site-packages/torch/profiler/profiler.py:224: Use

(APIServer pid=77603) INFO 08-25 17:46:15 [loggers.py:271] Engine 000: Avg prompt throughput: 238.3 tokens/s, Avg generation throughput: 195.2 tokens/s, Running: 16 reqs, Waiting: 68 reqs, GPU KV cache usage: 2.4%, Prefix cache hit rate: 0.0%
(APIServer pid=77603) INFO 08-25 17:46:25 [loggers.py:271] Engine 000: Avg prompt throughput: 242.7 tokens/s, Avg generation throughput: 195.1 tokens/s, Running: 16 reqs, Waiting: 52 reqs, GPU KV cache usage: 2.6%, Prefix cache hit rate: 0.0%
(APIServer pid=77603) INFO 08-25 17:46:35 [loggers.py:271] Engine 000: Avg prompt throughput: 240.0 tokens/s, Avg generation throughput: 198.4 tokens/s, Running: 15 reqs, Waiting: 36 reqs, GPU KV cache usage: 2.7%, Prefix cache hit rate: 0.0%
(APIServer pid=77603) INFO 08-25 17:46:45 [loggers.py:271] Engine 000: Avg prompt throughput: 490.1 tokens/s, Avg generation throughput: 198.4 tokens/s, Running: 16 reqs, Waiting: 4 reqs, GPU KV cache usage: 2.1%, Prefix cache hit rate: 0.0%
(APIServer pid=77603) INFO 08

(APIServer pid=77603) INFO:     Shutting down
(APIServer pid=77603) INFO:     Waiting for application shutdown.
(APIServer pid=77603) INFO:     Application shutdown complete.
(APIServer pid=77603) INFO:     Finished server process [77603]
/venv/main/lib/python3.12/multiprocessing/resource_tracker.py:279: UserWarning: resource_tracker: There appear to be 8 leaked semaphore objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '
/venv/main/lib/python3.12/multiprocessing/resource_tracker.py:279: UserWarning: resource_tracker: There appear to be 10 leaked shared_memory objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '


Server successfully shut down.


In [ ]:
results_imbalanced = await measure_vllm_throughput(model_path_imbalanced,
                                                             mmlu_prompts,
                                                             seed=seed,
                                                             max_new_tokens=max_new_tokens,
                                                             max_model_len=max_model_len,
                                                             batch_size=batch_size,
                                                             concurrency_limit=batch_size*4,
                                                             gpu_memory_utilization=gpu_memory_utilization,
                                                             n_gpus=n_gpus,
                                                             n_warmup_samples=n_warmup_samples,
                                                             print_output=False,
                                                             enable_expert_parallel=enable_expert_parallel,
                                                             enable_prefix_caching=enable_prefix_caching,
                                                             enable_bnb=enable_bnb,
                                                             trace_dir=trace_path_imbalanced)

Starting vLLM server for ./vllm_benchmarking/models/qwen-imbalance100...
Waiting for server to initialize ...
WARNING 08-25 17:56:34 [config.py:70] Support for Transformers v4 is deprecated. The Transformers v4 codepath will become unmaintained in vLLM v0.22.0 and will be removed in vLLM v0.24.0. Please upgrade to Transformers v5: pip install --upgrade transformers
WARNING 08-25 17:56:37 [profiler.py:129] Using 'torch' profiler with delay_iterations or max_iterations while ignore_frontend is False may result in high overhead.
(APIServer pid=78786) INFO 08-25 17:56:37 [utils.py:306] 
(APIServer pid=78786) INFO 08-25 17:56:37 [utils.py:306]        █     █     █▄   ▄█
(APIServer pid=78786) INFO 08-25 17:56:37 [utils.py:306]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.21.0
(APIServer pid=78786) INFO 08-25 17:56:37 [utils.py:306]   █▄█▀ █     █     █     █  model   ./vllm_benchmarking/models/qwen-imbalance100
(APIServer pid=78786) INFO 08-25 17:56:37 [utils.py:306]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(API

(APIServer pid=78786) The tokenizer you are loading from './vllm_benchmarking/models/qwen-imbalance100' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


WARNING 08-25 17:56:45 [config.py:70] Support for Transformers v4 is deprecated. The Transformers v4 codepath will become unmaintained in vLLM v0.22.0 and will be removed in vLLM v0.24.0. Please upgrade to Transformers v5: pip install --upgrade transformers
(EngineCore pid=78942) INFO 08-25 17:56:47 [core.py:109] Initializing a V1 LLM engine (v0.21.0) with config: model='./vllm_benchmarking/models/qwen-imbalance100', speculative_config=None, tokenizer='./vllm_benchmarking/models/qwen-imbalance100', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=8, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, str

(Worker pid=79120) (Worker pid=79099) (Worker pid=79105) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=79162) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=79135) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and 

(Worker pid=79094) INFO 08-25 17:57:14 [pynccl.py:111] vLLM is using nccl==2.28.9
(Worker pid=79094) WARNING 08-25 17:57:15 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=79162) WARNING 08-25 17:57:15 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=79148) WARNING 08-25 17:57:15 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=79120) WARNING 08-25 17:57:15 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=79105) (Worker pid=79112) WARNING 08-25 17:57:15 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
WARNING 08-25 17:57:15 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=79135) WARNING 08-

...

(Worker_TP7_EP7 pid=79162) (Worker_TP6_EP6 pid=79148) /venv/main/lib/python3.12/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
/venv/main/lib/python3.12/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
(Worker_TP5_EP5 pid=79135) (Worker_TP6_EP6 pid=79148) (Worker_TP7_EP7 pid=79162) /venv/main/lib/python3.12/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
  _warn_once(
(Worker_TP5_EP5 pid=79135)   _warn_once(
(Worker_TP2_EP2 pid=79105) /venv/main/lib/python3.12/site-pac

(APIServer pid=78786) INFO 08-25 17:58:24 [loggers.py:271] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 105.6 tokens/s, Running: 16 reqs, Waiting: 84 reqs, GPU KV cache usage: 2.9%, Prefix cache hit rate: 0.0%
(APIServer pid=78786) INFO 08-25 17:58:34 [loggers.py:271] Engine 000: Avg prompt throughput: 243.2 tokens/s, Avg generation throughput: 99.2 tokens/s, Running: 16 reqs, Waiting: 68 reqs, GPU KV cache usage: 2.4%, Prefix cache hit rate: 0.0%
(APIServer pid=78786) INFO 08-25 17:58:44 [loggers.py:271] Engine 000: Avg prompt throughput: 237.8 tokens/s, Avg generation throughput: 99.2 tokens/s, Running: 16 reqs, Waiting: 52 reqs, GPU KV cache usage: 2.0%, Prefix cache hit rate: 0.0%
(APIServer pid=78786) INFO 08-25 17:58:54 [loggers.py:271] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 102.4 tokens/s, Running: 16 reqs, Waiting: 52 reqs, GPU KV cache usage: 2.7%, Prefix cache hit rate: 0.0%
(APIServer pid=78786) INFO 08-25 1

(APIServer pid=78786) INFO:     Shutting down
(APIServer pid=78786) INFO:     Waiting for background tasks to complete. (CTRL+C to force quit)


(APIServer pid=78786) INFO 08-25 18:10:05 [launcher.py:137] Shutting down FastAPI HTTP server.


(APIServer pid=78786) INFO:     Shutting down
(APIServer pid=78786) INFO:     Waiting for background tasks to complete. (CTRL+C to force quit)


In [ ]:
results_imbalanced

In [ ]:
# Save results
import pickle
with open(results_file_balanced, 'wb') as file:
    pickle.dump(balanced_results, file)
with open(results_file_imbalanced, 'wb') as file:
    pickle.dump(imbalanced_results, file)

In [ ]:
def plot_forcedimbalance_results(results_balanced, results_imbalanced):

    # Get metrics: TTFT, TPOT
    ttfts_balanced = [r['ttft']*1000 for r in results_balanced]
    ttfts_imbalanced = [r['ttft']*1000 for r in results_imbalanced]
    tpots_balanced = [r['tpot']*1000 for r in results_balanced]
    tpots_imbalanced = [r['tpot']*1000 for r in results_imbalanced]
    all_ttfts = ttfts_balanced + ttfts_imbalanced
    all_tpots = tpots_balanced + tpots_imbalanced
    min_ttft = min(all_ttfts)
    min_tpot = min(all_tpots)
    max_ttft = max(all_ttfts)
    max_tpot = max(all_tpots)
    avg_ttfts_balanced = np.mean(ttfts_balanced)
    avg_ttfts_imbalanced = np.mean(ttfts_imbalanced)
    avg_tpots_balanced = np.mean(tpots_balanced)
    avg_tpots_imbalanced = np.mean(tpots_imbalanced)
    
    fig, axes = plt.subplots(2, figsize=(10, 10))
    plt.style.use('seaborn-v0_8-whitegrid')

    # Plot 1: TTFT
    ax = axes[0]
    # Create uniform bins
    n_bins = 100
    bin_width_ttft = (max_ttft - min_ttft) / n_bins
    bins_ttft = np.arange(min_ttft, max_ttft + bin_width_ttft, bin_width_ttft)
    
    # Hist for each
    ax.hist(ttfts_balanced, bins=bins_ttft,
                  label='baseline model')
    ax.hist(ttfts_imbalanced, bins=bins_ttft,
                  label='imbalanced model')

    ax.set_title('TTFT vs. Forced Imbalance')
    ax.set_xlabel('TTFT(ms)')
    ax.set_ylabel('Frequency')
    ax.legend()
    
    # Plot 2: TPOT
    ax = axes[1]
    # Create uniform bins
    n_bins = 100
    bin_width_tpot = (max_tpot - min_tpot) / n_bins
    bins_tpot = np.arange(min_tpot, max_tpot + bin_width_tpot, bin_width_tpot)
    
    # Hist for each
    ax.hist(tpots_balanced, bins=bins_tpot,
                  label='baseline model')
    ax.hist(tpots_imbalanced, bins=bins_tpot,
                  label='imbalanced model')

    ax.set_title('TPOT vs. Forced Imbalance')
    ax.set_xlabel('TPOT(ms)')
    ax.set_ylabel('Frequency')
    ax.legend()
    fig.suptitle('Performance Metrics from vLLM')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_forcedimbalance_results(results_balanced, results_imbalanced)